# Fashion-MNIST training: head-VM recovery and healthy overhead
Run the settings cell, then sections 1–6 and 8–10. Section 7 is for resuming an already prepared ML suite.
Two independent CNN training trials, one on each worker VM. Real Fashion-MNIST inputs; one CPU thread per trial; no GPU, sleep-based training, DDP, TorchFT, or checkpoint restart. Both tasks are owned on the original head. The driver and inputs survive off-head.

Every arm uses the same calibrated optimizer-step count, seeds, input bytes, placements and resources. The target is 90 seconds of actual training plus calibration headroom **per trial**; both measured trials must reach at least 60 seconds for a valid overhead sample. A shorter run is retained and excluded, not padded with extra work. Default: one full five-case comparison; set REPEATS=2 for a longer resumable suite.

Use EXISTING_STATE_FILE to reuse your current cluster, or None to create fresh VMs. Preparation can revive only the stopped original head after checking for active services/locks. It never deletes VMs. Each run/resume invocation has a 28-minute work budget plus 90-second cleanup; the matrix can span invocations. Download results before stopping/deleting VMs.

The assistant has not executed any of these cells. See [ML experiment details](head_vm_loss/ML_PILOT.md).


In [ ]:
import concurrent.futures
import json
import shlex
import subprocess
import sys
import time
import uuid
from pathlib import Path

PROJECT = "research-488322"
ZONE = "us-central1-a"
IMAGE = "ray-built-v1"
NETWORK = "default"
SUBNET = "default"
MACHINE_TYPE = "e2-standard-4"  # The current image check requires four CPUs.
DISK_GB = 100
PYTHON = "/opt/ray-env/ray-dev/bin/python"

# None creates fresh VMs. To reuse an existing cluster, enter its state.json path.
EXISTING_STATE_FILE = None  # Or Path("/absolute/path/to/state.json")
SUITE = "fashion-head-v1"
MAX_EXPERIMENT_MINUTES = 28  # Per run/resume invocation; cleanup reserves 90 seconds.
KEEPALIVE_INTERVAL_MS = 10000  # Same in every arm and on every Ray process.
KEEPALIVE_TIMEOUT_MS = 10000
REPEATS = 1  # 1 or 2 full matched repetitions; same suite resumes recorded progress.
ML_SETTINGS = dict(
    target_training_s=90,        # Calibration prediction, not a measured result.
    minimum_training_s=60,       # Measured optimizer-loop wall time, per trial.
    calibration_s=10,            # Real training, outside experimental observations.
    batch_size=128,              # 32..256
    train_examples=10000,        # First 10,000 training images.
    validation_examples=2000,   # Next 2,000 training images; model selection only.
    test_examples=2000,         # First 2,000 official test images.
    seed=2026,
    learning_rates=[0.001, 0.0003],
    minimum_accuracy=0.65,      # Sanity check; exact healthy-reference hashes also checked.
)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "gossip_benchmarks").is_dir())
HELPERS = ROOT / "gossip_benchmarks/selective_replication"
NATIVE_HELPERS = ROOT / "gossip_benchmarks/head_vm_loss"
WORKDIR = ROOT / "ray_gcp_results"
sys.path.insert(0, str(HELPERS))
sys.path.insert(0, str(NATIVE_HELPERS))

if EXISTING_STATE_FILE is not None:
    STATE_FILE = Path(EXISTING_STATE_FILE).expanduser().resolve()
    state = json.loads(STATE_FILE.read_text())
    PROJECT, ZONE, RUN_ID = state["project"], state["zone"], state["run_id"]
    NAMES = state["names"]
else:
    if "RUN_ID" not in globals():
        RUN_ID = "rayexp-" + time.strftime("%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:6]
    NAMES = [RUN_ID + "-head"] + [RUN_ID + f"-w{i}" for i in range(1, 5)]
    STATE_FILE = WORKDIR / RUN_ID / "state.json"
LOCAL = WORKDIR / RUN_ID
LOCAL.mkdir(parents=True, exist_ok=True)
HEAD, *WORKERS = NAMES
REGION = ZONE.rsplit("-", 1)[0]
print("VMs:", *NAMES, sep="\n  ")
print("State file:", STATE_FILE)


## 1. Small infrastructure helpers
Only infrastructure cells use these gcloud wrappers. They share a 28-minute setup budget starting with the first cloud command; pauses count. Preparation and each pilot invocation have separate bounded budgets, so initial provisioning plus the full matrix can take more than 30 minutes. No experiment starts automatically.


In [ ]:
def gcloud(*args, timeout=600):
    global EXPERIMENT_DEADLINE
    if "EXPERIMENT_DEADLINE" not in globals():
        EXPERIMENT_DEADLINE = time.monotonic() + 28 * 60
    if not globals().get("CLEANUP_MODE", False):
        remaining = EXPERIMENT_DEADLINE - time.monotonic()
        if remaining <= 0:
            raise TimeoutError("Infrastructure setup budget exhausted. Inspect saved state before continuing.")
        timeout = min(timeout, remaining)
    command = ["gcloud", *map(str, args), f"--project={PROJECT}", "--quiet"]
    print(shlex.join(command), flush=True)
    result = subprocess.run(command, text=True, capture_output=True, timeout=timeout)
    if result.returncode:
        raise RuntimeError(result.stderr + "\n" + result.stdout)
    return result.stdout


def ssh(name, command):
    return gcloud("compute", "ssh", name, f"--zone={ZONE}", "--tunnel-through-iap",
                  "--ssh-flag=-oConnectTimeout=15", f"--command={command}")


def scp(source, target):
    return gcloud("compute", "scp", source, target, f"--zone={ZONE}", "--tunnel-through-iap")


def run_parallel(function, items):
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as pool:
        return list(pool.map(function, items))


## 2. Inspect current instances — read only
Do not operate this cluster while another notebook is using it. Existing VMs must match the recorded ownership labels.


In [ ]:
inventory = json.loads(gcloud("compute", "instances", "list", "--format=json"))
for vm in inventory:
    if vm.get("labels", {}).get("ray-experiment"):
        print(vm["name"], vm["status"], vm["zone"].rsplit("/", 1)[-1])


## 3. Create fresh VMs, or skip for an existing state file
Uses the existing default VPC/subnet. No network is created. Names are recorded before allocation; after a partial creation failure inspect the state rather than silently creating a second cluster.


In [ ]:
from gcp.capabilities import stop_command

print("Failure command:", "gcloud", *stop_command(), "--no-graceful-shutdown")
if STATE_FILE.exists():
    print("Creation already recorded for this session. Continue with setup.", STATE_FILE)
else:
    image = json.loads(gcloud("compute", "images", "describe", IMAGE, "--format=json"))
    assert image["status"] == "READY"
    gcloud("compute", "networks", "describe", NETWORK, "--format=json")
    subnet = json.loads(gcloud("compute", "networks", "subnets", "describe", SUBNET,
                               f"--region={REGION}", "--format=json"))
    assert subnet["network"].rsplit("/", 1)[-1] == NETWORK, "Subnet belongs to a different VPC"
    # Record names before creation so partial failures can be cleaned up.
    STATE_FILE.write_text(json.dumps(dict(project=PROJECT, zone=ZONE, run_id=RUN_ID,
                                         names=NAMES, image=IMAGE, network=NETWORK, subnet=SUBNET), indent=2))
    gcloud("compute", "firewall-rules", "create", RUN_ID + "-internal",
           f"--network={NETWORK}", "--allow=tcp,udp,icmp",
           f"--source-tags={RUN_ID}", f"--target-tags={RUN_ID}")
    gcloud("compute", "firewall-rules", "create", RUN_ID + "-iap",
           f"--network={NETWORK}", "--allow=tcp:22",
           "--source-ranges=35.235.240.0/20", f"--target-tags={RUN_ID}")
    gcloud("compute", "instances", "create", *NAMES, f"--zone={ZONE}",
           f"--machine-type={MACHINE_TYPE}", f"--image={IMAGE}", f"--image-project={PROJECT}",
           f"--boot-disk-size={max(DISK_GB, int(image['diskSizeGb']))}GB",
           "--boot-disk-type=pd-balanced", "--provisioning-model=STANDARD",
           f"--subnet={SUBNET}", f"--tags={RUN_ID}", f"--labels=ray-experiment={RUN_ID}",
           "--metadata=enable-oslogin=FALSE", "--no-service-account", "--no-scopes", timeout=900)


## 4. Get IPs and wait for SSH
For a resume after the last head-loss observation, skip creation, node setup and preparation. Reconnect to the saved preparation in section 7 instead.


In [ ]:
nodes = []
for name in NAMES:
    vm = json.loads(gcloud("compute", "instances", "describe", name, f"--zone={ZONE}", "--format=json"))
    assert vm.get("labels", {}).get("ray-experiment") == RUN_ID, name
    assert vm["status"] == "RUNNING", (name, vm["status"])
    nodes.append(dict(name=name, ip=vm["networkInterfaces"][0]["networkIP"], head=name == HEAD))
HEAD_IP = nodes[0]["ip"]
print(json.dumps(nodes, indent=2))


def wait_ssh(name):
    for attempt in range(30):
        try:
            ssh(name, "true")
            return
        except (RuntimeError, subprocess.TimeoutExpired):
            if attempt == 29:
                raise
            time.sleep(5)


wait_ssh(HEAD)  # Initialize gcloud's local SSH key before parallel connections.
run_parallel(wait_ssh, WORKERS)


## 5. Install the visible node helper
The image already contains Ray/PyTorch. This sets up the experiment account and node identity files. It does not create head-hosted NFS: surviving storage is prepared on w4 next.


In [ ]:
def setup_node(node):
    name = node["name"]
    ssh(name, """set -e
sudo apt-get update -qq
sudo DEBIAN_FRONTEND=noninteractive apt-get -o DPkg::Lock::Timeout=120 install -y -qq nfs-common
id raybench >/dev/null 2>&1 || sudo useradd --uid 25001 --create-home --home-dir /var/lib/raybench --shell /bin/bash raybench
test "$(id -u raybench)" = 25001
sudo mkdir -p /opt/ray-gcp /srv/ray-gcp
mountpoint -q /srv/ray-gcp || sudo chown raybench:raybench /srv/ray-gcp
""")
    path = LOCAL / (name + ".json")
    path.write_text(json.dumps(dict(node, nodes=nodes, head_ip=HEAD_IP, run_id=RUN_ID)))
    scp(path, name + ":/tmp/ray-node.json")
    scp(HELPERS / "gcp/bootstrap/node_control.py", name + ":/tmp/ray-node-control.py")
    ssh(name, """set -e
sudo install -o root -g root -m 644 /tmp/ray-node.json /etc/ray-gcp.json
sudo install -o root -g root -m 644 /tmp/ray-node-control.py /opt/ray-gcp/node_control.py
""")


run_parallel(setup_node, nodes)


## 6. Prepare cluster, inputs and fixed training work — initial ML setup
Reuse requires the existing state file in settings. If only the original head is stopped, this cell checks for active experiment services and the suite lock before starting it. All five VMs must then be running. Fresh clusters use the same preparation.

Downloads Fashion-MNIST once on w4 via the publisher's HTTPS mirror, retaining torchvision checksum validation; copies a fixed input file to each training VM; then performs a short CPU training calibration on each. This requires outbound HTTPS only during preparation. Dataset download, copying and calibration are excluded from experiment timings.

Calibration is saved locally and reused on resume. Every arm executes the same steps. Do not change ML_SETTINGS or recalibrate midway through a suite. If either trial later falls below 60 seconds, increase the target and use a NEW suite. The workload is visible in native_ml_task.py and native_ml_workload.py.


In [ ]:
from native_ml_pilot import NativeMLPilot

cloud = NativeMLPilot(STATE_FILE, WORKDIR / "head-vm-loss")
environment_file = cloud.local / "environment.json"
if environment_file.exists():
    cloud.nodes = json.loads(environment_file.read_text())["nodes"]

# New experiment setup only: never revive the failed head during failure cleanup.
if cloud.describe(cloud.head)["status"] == "TERMINATED":
    if cloud.ssh(cloud.support, "test ! -d /var/lib/raybench/ray-headloss-lock", check=False).returncode:
        raise RuntimeError("Controller lock exists; inspect cleanup before starting the head")
    for name in cloud.names:
        if name != cloud.head:
            active = cloud.ssh(name, "systemctl list-units 'ray-headloss-*' 'ray-native-*' "
                                    "--state=active,activating --no-legend").stdout
            if active.strip():
                raise RuntimeError("Active services must be inspected: " + active)
    cloud.gc("compute", "instances", "start", cloud.head, "--zone=" + cloud.zone, timeout=180)
    boot_deadline = time.monotonic() + 120
    while cloud.ssh(cloud.head, "true", check=False, timeout=20).returncode:
        if time.monotonic() > boot_deadline:
            raise TimeoutError("Original head did not become SSH-reachable")
        time.sleep(2)

cloud.prepare()
preparation = cloud.prepare_ml(SUITE, ML_SETTINGS)
print("Fixed steps per trial:", preparation["ml"]["steps"])
print("Predicted training seconds (calibration only):", preparation["predicted_training_s"])
print("Dataset:", preparation["dataset"])
print("Prepared:", cloud.local)


## 7. Reconnect after restarting the kernel — optional
Skip if section 6 just succeeded. For a normal resume, run settings with the same EXISTING_STATE_FILE, SUITE, ML_SETTINGS and REPEATS; set RESUME_PREPARED=True below, then run section 8. Do not redo preparation or calibration during a suite. The saved native helpers, calibration, settings and VM identities are checked.


In [ ]:
RESUME_PREPARED = False
if RESUME_PREPARED:
    from native_ml_pilot import NativeMLPilot
    from gcp.capabilities import stop_command

    cloud = NativeMLPilot(STATE_FILE, WORKDIR / "head-vm-loss")
    environment = json.loads((cloud.local / "environment.json").read_text())
    cloud.nodes = environment["nodes"]
    cloud.stop_args = stop_command()
    cloud.ready = True
    print("Using saved preparation:", cloud.local)


## 8. Run or resume the ML comparison
Five cases per repetition: plain healthy; Redis/proxy healthy with Fixed-R OFF and ON; identical head-VM loss with Fixed-R OFF and ON. Training is paused at 10% of the fixed steps in every arm. Healthy arms release the tasks; fault arms stop the head VM and leave original gates closed. Replayed normal tasks train from the same seed and inputs, without a checkpoint.

The 28-minute invocation may pause before completing all cases. Rerun only this cell to continue. Failed observations remain recorded. Setup errors still require inspection. Healthy overhead includes startup, training, evaluation and weight transfer; the plot also reports optimizer-loop time separately. Both successful trials must have at least 60 measured training seconds to enter an overhead comparison.


In [ ]:
summary = cloud.run_ml(
    SUITE,
    ML_SETTINGS,
    repeats=REPEATS,
    budget_s=MAX_EXPERIMENT_MINUTES * 60,
    keepalive_interval_ms=KEEPALIVE_INTERVAL_MS,
    keepalive_timeout_ms=KEEPALIVE_TIMEOUT_MS,
)
print("Finished matrix:", summary.get("finished_matrix", False))
print("Pause reason:", summary.get("paused_reason"))
print("Cleanup errors:", summary.get("cleanup_errors", []))


## 9. Inspect saved progress — no cloud calls
Failures before injection are setup failures, not evidence of failed recovery.


In [ ]:
REPORT = cloud.local / SUITE / "native-report.json"
report = json.loads(REPORT.read_text())
print("Recorded:", len(report["samples"]), "/", report["planned_observations"])
for row in report["samples"]:
    result = row.get("result", {})
    print(row["label"], row["status"], "correct:", result.get("correct", False),
          "elapsed:", result.get("elapsed_s"),
          "valid overhead sample:", result.get("valid_overhead_sample", False))
    for trial in result.get("trials", []):
        metrics = trial["metrics"]
        print("  trial", metrics["index"], "training seconds:", round(trial["training_s"], 3),
              "steps:", metrics["steps"], "validation accuracy:", round(metrics["validation_accuracy"], 4),
              "test accuracy:", round(metrics["test_accuracy"], 4))
    if row.get("error") or result.get("error") or result.get("duration_warning"):
        print(" ", row.get("error") or result.get("error") or result.get("duration_warning"))
    if row["fault"]:
        print("  reexecutions:", result.get("reexecutions"),
              "abandoned optimizer steps:", result.get("abandoned_optimizer_steps"),
              "resumption upper bound:", row.get("failure_to_resumption_upper_bound_s"),
              "head stayed off:", row.get("original_head_remained_off"))
print("Results:", REPORT.parent)


## Plot saved results — run this cell alone
Set RESULTS_DIR to a saved suite folder, or leave None when cloud and SUITE exist in this kernel. No preparation, SSH or reruns. Failed outcomes stay visible. Points only, no bars or bottom annotations. Timing changes from one replicate are preliminary.


In [ ]:
from pathlib import Path
import json
import statistics
import matplotlib.pyplot as plt

RESULTS_DIR = None  # Or Path("/absolute/path/to/fashion-head-v1")
if RESULTS_DIR is None:
    if "cloud" not in globals() or "SUITE" not in globals():
        raise ValueError("Set RESULTS_DIR to your saved ML suite folder")
    RESULTS_DIR = cloud.local / SUITE
report_path = Path(RESULTS_DIR).expanduser() / "native-report.json"
report = json.loads(report_path.read_text())
rows = report["samples"]
by_label = {r["label"]: r for r in rows}

for baseline, candidate, title in [
    ("plain-healthy", "off-healthy", "Redis/proxy overhead"),
    ("off-healthy", "on-healthy", "Additional Fixed-R overhead"),
    ("plain-healthy", "on-healthy", "Full recovery-enablement overhead"),
]:
    for metric, metric_name in [
        ("elapsed_excluding_supervisor_gate_s", "job time excluding supervisor wait"),
        ("training_critical_s", "optimizer-loop wall time (slower trial)"),
    ]:
        changes = []
        for repeat in range(2):
            a = by_label.get(f"{baseline}-r{repeat}", {}).get("result", {})
            b = by_label.get(f"{candidate}-r{repeat}", {}).get("result", {})
            if a.get("correct") and b.get("correct") and a.get("valid_overhead_sample") and b.get("valid_overhead_sample"):
                delta = 100 * (b[metric] / a[metric] - 1)
                changes.append(delta)
                print(f"{title}, {metric_name}, repeat {repeat}: {delta:+.2f}% "
                      f"({a[metric]:.3f}s -> {b[metric]:.3f}s)")
        if len(changes) > 1:
            print(f"  Mean paired change: {statistics.mean(changes):+.2f}%; "
                  f"range {min(changes):+.2f}% to {max(changes):+.2f}%")
        elif not changes:
            print(title + ", " + metric_name + ": no valid matched pair yet")

healthy = [r for r in rows if not r["fault"]]
faults = [r for r in rows if r["fault"]]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, metric, title in [
    (axes[0], "elapsed_excluding_supervisor_gate_s", "Healthy job time, supervisor wait excluded (s)"),
    (axes[1], "training_critical_s", "Actual training, slower trial (s)"),
]:
    for i, row in enumerate(healthy):
        result = row.get("result", {})
        if result.get("correct") and result.get("valid_overhead_sample"):
            ax.scatter(result[metric], i)
        else:
            ax.text(.02, i, "Failed or below 60s", transform=ax.get_yaxis_transform(), va="center", color="tab:red")
    ax.set_yticks(range(len(healthy)), [r["label"] for r in healthy])
    ax.invert_yaxis()
    ax.set_title(title)
for i, row in enumerate(faults):
    value = row.get("failure_to_resumption_upper_bound_s")
    if value is not None:
        axes[2].scatter(value, i, color="tab:orange")
    else:
        axes[2].text(.02, i, "No verified resumption", transform=axes[2].get_yaxis_transform(), va="center", color="tab:red")
axes[2].set_yticks(range(len(faults)), [r["label"] + " (" + r["status"] + ")" for r in faults])
axes[2].invert_yaxis()
axes[2].set_title("Observed training-resumption upper bound (s)")
fig.tight_layout()
fig.savefig(report_path.parent / "fashion-head-overhead.png", dpi=160)
plt.show()


## 10. Collect evidence before deleting VMs
Copies native log tails, task attempt evidence and Redis state from w4. Keep the local native-report.json as well. This cell does not rerun workloads. If collection fails, keep w4 and its disk until you have the evidence.


In [ ]:
print("Local results:", cloud.local / SUITE)
DOWNLOAD_EVIDENCE = True
if DOWNLOAD_EVIDENCE:
    evidence_dir = cloud.download(SUITE)
    print("Upload:", cloud.local / SUITE / "native-report.json")
    print("Upload:", evidence_dir / "evidence.tar.gz")
print("Calibration metadata:", cloud.local / ("ml-preparation-" + SUITE + ".json"))


## Diagnose an IAP/SSH failure — optional
After a connection error, restart the kernel and run the settings cell with EXISTING_STATE_FILE pointing to this cluster's saved state.json. Set DIAGNOSE_CONNECTIONS=True below. This cell checks VM identity/status, SSH reachability, support services and the suite lock; it changes no cloud state.

An IAP 4003 / port 22 error is a controller transport failure, not evidence of Ray recovery failure. The native runner now retries that specific connection-creation error up to three times within its original command/session budget. Other errors and timeouts are not blindly retried. Events are saved in native-transport.jsonl and each observation's transport_events.

Keep the old suite/report. If surviving VMs are RUNNING, SSH is reachable, cleanup succeeded, and no suite lock remains, set SUITE="fashion-head-v1", run section 6 to upload the updated helpers, then section 8. Do not use section 7 to bypass the helper update. If cleanup errors or a lock remain, inspect them before proceeding; this notebook does not automatically remove a lock. You do not need to recreate a healthy cluster solely because a connection failed.


In [ ]:
# Run independently after the settings cell. Does not start/stop VMs or Ray.
DIAGNOSE_CONNECTIONS = False
PREVIOUS_SUITE = "native-head-v5"

if DIAGNOSE_CONNECTIONS:
    from native_pilot import NativePilot

    diagnostic_cloud = NativePilot(STATE_FILE, WORKDIR / "head-vm-loss")
    environment_path = diagnostic_cloud.local / "environment.json"
    if environment_path.exists():
        diagnostic_cloud.nodes = json.loads(environment_path.read_text())["nodes"]
    previous_report = diagnostic_cloud.local / PREVIOUS_SUITE / "native-report.json"
    if previous_report.exists():
        previous = json.loads(previous_report.read_text())
        print("Previous cleanup errors:", previous.get("cleanup_errors", "not recorded"))
        for row in previous.get("samples", []):
            print(row["label"], row["status"])
    for name in diagnostic_cloud.names:
        try:
            vm = diagnostic_cloud.describe(name)
            print(name, vm["status"])
            if vm["status"] == "RUNNING":
                probe = diagnostic_cloud.ssh(name, "true", check=False, timeout=30)
                print("  SSH:", "reachable" if probe.returncode == 0 else probe.stderr[-2000:])
            if name == diagnostic_cloud.support and vm["status"] == "RUNNING":
                probe = diagnostic_cloud.ssh(name, """uptime
systemctl is-active ssh
systemctl list-units 'ray-headloss-*' 'ray-native-*' --state=active,activating --no-legend
if test -d /var/lib/raybench/ray-headloss-lock; then
    printf 'Suite lock is present; inspect cleanup before proceeding.\\n'
else
    printf 'No suite lock.\\n'
fi
""", check=False, timeout=30)
                print(probe.stdout, probe.stderr)
        except Exception as exc:
            print(name, str(exc))


## 11. Optional VM stop
Stops only the five recorded, ownership-checked VMs. Disks/evidence remain allocated; no deletion. Download wanted results first. Do not stop a cluster with a live controller or unresolved cleanup errors.


In [ ]:
STOP_VMS = False
if STOP_VMS:
    if cloud.ssh(cloud.support, "test ! -d /var/lib/raybench/ray-headloss-lock", check=False).returncode:
        raise RuntimeError("Suite lock exists; inspect the controller before stopping VMs")
    for name in cloud.names:
        vm = cloud.describe(name)
        if vm["status"] == "RUNNING":
            cloud.gc("compute", "instances", "stop", name, "--zone=" + cloud.zone, timeout=300)
else:
    print("VM stop skipped. Running VMs and persistent disks remain billable.")
